# Transforms: Composable Mappings of Fields

> **tldr:** `Transform` is Terrax's standard abstraction for data processing, defining composable mappings between dictionaries of `coordax.Field` objects. In this guide, you will learn how transforms work and how to assemble reusable building blocks into complex feature extraction and normalization pipelines.

**Key concepts:**
* **Core API:** `Transform` couples runtime data processing (`__call__`) with shapes and coordinates inference (`output_shapes`)
* **Keys & Values:** Transforms can operate on dictionary structure by modifying keys, as well as data by modifying fields
* **Composition:** Transforms can be assembled in sequence (`Sequential`) and over parallel branches (`Merge`)
* **Applications:** Together, they enable clean, highly modular feature extraction and data processing pipelines


In [ ]:
import warnings
warnings.simplefilter('ignore')

import coordax as cx
from flax import nnx
import jax.numpy as jnp
import jax_datetime as jdt
from neuralgcm.experimental.core import coordinates
from neuralgcm.experimental.core import feature_transforms
from neuralgcm.experimental.core import random_processes
from neuralgcm.experimental.core import spherical_harmonics
from neuralgcm.experimental.core import transforms
import numpy as np

## Transform Protocol

`Transform` instances map between dictionaries of `coordax.Field` objects, which is the core [data model](https://neuralgcm.readthedocs.io/en/latest/data_model_intro.html) in Terrax used to represent simulation state and observations.

`Transform` instances define a `__call__` and `output_shapes` methods with the following signatures:
```python
def __call__(self, inputs: dict[str, cx.Field]) -> dict[str, cx.Field]:
  ...

def output_shapes(
    self, input_shapes: dict[str, cx.Field]
) -> dict[str, cx.Field]:
  ...
```
These two methods implement the core logic and enable efficient shape inference that facilitates instantiation of modules that accept Transform's outputs.


Transforms generally fall into three categories:
1. **Transforms that modify keys**: Selecting or renaming entries
2. **Transforms that modify values**: Operations on the data within the fields
3. **Compositions and wrappers**: Combining multiple transforms or modifiers

In the following subsections, we will introduce examples for each category.

### Transforms that modify keys

These transforms focus on changing the structure of the dictionary keeping associated values unchanged. Examples include:
* `SelectKeys` - subselects keys in the dictionary
* `FilterByCoord` - subselects keys based on coordinates of the corresponding `Field` value
* `RenameKeys` - renames dictionary keys

Let's look at `SelectKeys` and `RenameKeys` in action.

In [3]:
x = cx.SizedAxis('x', 3)
inputs = {
    'temperature': cx.field(jnp.array([1.0, 2.0, 3.0]), x),
    'wind': cx.field(jnp.array([4.0, 5.0, 6.0]), x),
    'time': cx.field(0.0),
}

select_transform = transforms.SelectKeys(['temperature', 'time'])
selected_outputs = select_transform(inputs)
print('Selected keys:', selected_outputs.keys())

rename_transform = transforms.RenameKeys({'temperature': 'T'})
renamed_outputs = rename_transform(selected_outputs)
print('Renamed keys:', renamed_outputs.keys())

Selected keys: dict_keys(['temperature', 'time'])
Renamed keys: dict_keys(['T', 'time'])


### Transforms that modify values

These transforms update `cx.Field` values, generally leaving the dict structure unaffected, unless required by the transform

Here we demonstrate `ScaleFields`, `FillNaNs`, and `ApplyFnToKeys`.

In [ ]:
x = cx.SizedAxis('x', 3)
inputs_with_nans = {
    'a': cx.field(jnp.array([1.0, jnp.nan, 3.0]), x),
    'b': cx.field(jnp.array([4.0, 5.0, 6.0]), x),
}

scale_transform = transforms.ScaleFields(cx.field(2.0))
scaled_outputs = scale_transform(inputs_with_nans)
print('Scaled outputs \'a\':', scaled_outputs['a'].data)

fill_nan_transform = transforms.FillNaNs(value=0.0)
filled_outputs = fill_nan_transform(inputs_with_nans)
print('Filled outputs \'a\':', filled_outputs['a'].data)

apply_fn_transform = transforms.ApplyFnToKeys(cx.cmap(jnp.sin), ['b'])
fn_outputs = apply_fn_transform(inputs_with_nans)
print('ApplyFn outputs \'b\' (sin):', fn_outputs['b'].data)

Scaled outputs 'a': [ 2. nan  6.]
Filled outputs 'a': [1. 0. 3.]
ApplyFn outputs 'b' (sin): [-0.7568025  -0.9589243  -0.27941552]


## 2. Composition

Individual transforms become far more powerful when combined. Terrax provides built-in combinators to construct complex data processing pipelines out of modular building blocks:

*   `Sequential`: Chains multiple transforms together in order
*   `Merge`: Applies multiple parallel transforms to the same input and gathers their results
*   `ApplyToKeys` (and similar): Syntactic sugar for common composition and subsetting patterns

Below, we demonstrate how to chain the transforms introduced earlier into a single pipeline using `Sequential` and `Merge`.


In [5]:
# Create some sample data again
x = cx.SizedAxis('x', 3)
inputs = {
    'temperature': cx.field(jnp.array([1.0, 2.0, 3.0]), x),
    'wind': cx.field(jnp.array([4.0, 5.0, 6.0]), x),
    'time': cx.field(0.0),
}

# Define a sequential transform
sequential_transform = transforms.Sequential([
    # 1. Select only 'temperature' and 'time'
    transforms.SelectKeys(['temperature', 'time']),
    # 2. Rename 'temperature' to 'T'
    transforms.RenameKeys({'temperature': 'T'}),
    # 3. Scale all remaining fields (T and time) by 10
    transforms.ScaleFields(cx.field(10.0)),
])

outputs = sequential_transform(inputs)
print('Outputs keys:', outputs.keys())
print('Outputs \'T\' data:', outputs['T'].data)
print('Outputs \'time\' data:', outputs['time'].data)

Outputs keys: dict_keys(['T', 'time'])
Outputs 'T' data: [10. 20. 30.]
Outputs 'time' data: 0.0


### Branching and Merging with `Merge`

`Merge` allows you to run multiple transforms on the same input and combine their results. This is useful for generating different sets of features in parallel. You can also specify prefixes to avoid key collisions.

In [6]:
# Create some sample data again to ensure keys exist
x = cx.SizedAxis('x', 3)
inputs = {
    'a': cx.field(jnp.array([1.0, 2.0, 3.0]), x),
    'b': cx.field(jnp.array([4.0, 5.0, 6.0]), x),
}

# Create two branches
branch1 = transforms.SelectKeys('a')
branch2 = transforms.Sequential([
    transforms.SelectKeys('b'),
    transforms.ScaleFields(cx.field(10.0)),
])

# Merge them, adding prefixes
merged_transform = transforms.Merge(
    {'original_a': branch1, 'scaled_b': branch2},
    always_add_prefix=True
)

merged_outputs = merged_transform(inputs)
print('Merged outputs keys:', merged_outputs.keys())

Merged outputs keys: dict_keys(['original_a_a', 'scaled_b_b'])



### Helper Functions: `ApplyToKeys` vs Manual Branching
Helper transforms like `ApplyToKeys` serve as convenient syntactic sugar for common branching patterns. While applying an operation to a subset of dictionary keys can always be implemented manually by combining `SelectKeys`, `Sequential`, and `Merge`, `ApplyToKeys` makes this intent clear and concise.
Below, we compare the explicit manual branching implementation against its streamlined `ApplyToKeys` equivalent.

In [ ]:
# Create some sample data
x = cx.SizedAxis('x', 3)
inputs = {
    'a': cx.field(jnp.array([1.0, 2.0, 3.0]), x),
    'b': cx.field(jnp.array([4.0, 5.0, 6.0]), x),
}

# Operation to apply: Scale by 10
scale_10 = transforms.ScaleFields(cx.field(10.0))

# 1. Manual Branching
# Select 'a', scale it, and merge with original 'b'
manual_branch = transforms.Merge({
    'scaled_a': transforms.Sequential([transforms.SelectKeys('a'), scale_10]),
    'keep_b': transforms.SelectKeys('b'),
}, always_add_prefix=False)

outputs_manual = manual_branch(inputs)
print('Manual branching outputs keys:', outputs_manual.keys())

# 2. Using ApplyToKeys
apply_to_keys = transforms.ApplyToKeys(transform=scale_10, keys=['a'])

outputs_apply = apply_to_keys(inputs)
print('ApplyToKeys outputs keys:', outputs_apply.keys())

Manual branching outputs keys: dict_keys(['a', 'b'])
ApplyToKeys outputs keys: dict_keys(['a', 'b'])


## 3. Usage Example 1: Divergence and Vorticity to Velocity

In this example, we will use `VelocityFromDivCurl` transform to compute horizontal velocity components ($u$, $v$) from divergence and vorticity repesentation. This shows that transform can modify both keys, values and associated coordinates.

There are many other similar transformations that focus on regridding, interpolation, inpainting and so on.


In [8]:
# Create a grid and mapping
nodal_grid = coordinates.LonLatGrid.T21()
modal_grid = coordinates.SphericalHarmonicGrid.T21()
ylm_map = spherical_harmonics.FixedYlmMapping(nodal_grid, modal_grid)

# Use randomness process to generate modal divergence and vorticity
rngs = nnx.Rngs(0)

# Instantiate random processes
random_div = random_processes.NormalUncorrelated(mean=0.0, std=1.0, coord=modal_grid, rngs=rngs)
random_vor = random_processes.NormalUncorrelated(mean=0.0, std=1.0, coord=modal_grid, rngs=rngs)

inputs = {
    'divergence': random_div.state_values(),
    'vorticity': random_vor.state_values(),
}

# Transform to velocity
to_velocity = transforms.VelocityFromDivCurl(ylm_map)
outputs = to_velocity(inputs)

print('Outputs keys:', outputs.keys())
print('U component shape:', outputs['u_component_of_wind'].shape)
print('V component shape:', outputs['v_component_of_wind'].shape)

Outputs keys: dict_keys(['u_component_of_wind', 'v_component_of_wind'])
U component shape: (64, 32)
V component shape: (64, 32)


## 4. Usage Example 2: Feature Engineering and Normalization

In deep learning applications standardizing inputs is often mandatory for stable training. In this example, we use `StreamNorm` transform to scale raw inputs to zero mean and unit variance.

`StreamNorm` is an example of a stateful tranform that maintains streaming statistics for normalization and updates it when update_stats is set to `True`.
A common strategy is to use a fixed set of batches to estimate normalization constants and freeze them for the remainer of training.

In [ ]:
# Create a grid
grid = coordinates.LonLatGrid.T21()

# Create dummy inputs with time using jax_datetime
inputs = {
    'time': cx.field(jdt.to_datetime('2026-04-21T00:00')),
}

# Instantiate feature transforms
lat_features = feature_transforms.LatitudeFeatures(grid=grid)
radiation_features = feature_transforms.RadiationFeatures(grid=grid)

# Merge them
feature_pipeline = transforms.Merge({
    'lat_lon': lat_features,
    'radiation': radiation_features,
})

all_features = feature_pipeline(inputs)

# Now let's normalize them using StreamNorm
norm = transforms.StreamNorm(
    norm_coords={
        'radiation': cx.Scalar(),
        'cos_latitude': cx.Scalar(),
        'sin_latitude': cx.Scalar(),
    },
    update_stats=True,
    epsilon=1e-11,
)

# Update stats with current features
_ = norm(all_features)

# Apply normalization
norm.update_stats = False
normalized_features = norm(all_features)

print('Normalized features keys:', normalized_features.keys())
print('Normalized radiation shape:', normalized_features['radiation'].shape)
print(np.mean(normalized_features['radiation'].data))
print(np.std(normalized_features['radiation'].data))

Normalized features keys: dict_keys(['cos_latitude', 'radiation', 'sin_latitude'])
Normalized radiation shape: (64, 32)
